# ChemFF Metrics — For Progress Report\n\nThis notebook trains the ChemFF (Chemprop D-MPNN + Morgan FP) hybrid model\nand a Chemprop-only baseline, then outputs all metrics needed for the\nprogress report LaTeX tables.\n\n**Run this on Colab with A100 for fastest results.**\n\nOutputs:\n- Chemprop-only baseline: R², RMSE, Pearson r, Top-10%, Top-20%\n- Chemprop + Morgan hybrid: R², RMSE, Pearson r, Top-10%, Top-20%\n- Comparison table (copy-paste into LaTeX)

In [ ]:
# ── Cell 1: Setup (Colab) ─────────────────────────────────────────────────
import os, sys

# --- Google Colab setup ---
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/LANTERN"
    os.chdir(PROJECT)
    !pip install -q rdkit-pypi
else:
    PROJECT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    os.chdir(PROJECT)

sys.path.insert(0, PROJECT)
print(f"Working directory: {os.getcwd()}")
print(f"Project root: {PROJECT}")

In [ ]:
# ── Cell 2: Imports & Device ──────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from copy import deepcopy
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from torch.utils.data import DataLoader, Subset
from torch.amp import autocast, GradScaler

from chemff.model import ConcatFFN
from chemff.dataset import MoleculeDataset, collate_fn, load_dataset_from_config
from chemff.chemprop.featurizer import MolecularGraphFeaturizer
from chemff.morgan.featurizer import MorganFeaturizer

# ── Device detection ──
if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_properties(0).name
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    # Enable TF32 for A100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    device = "cpu"
    print("Running on CPU (will be slow)")

print(f"Device: {device}")

In [ ]:
# ── Cell 3: Load config & data ────────────────────────────────────────────
with open("chemff/config.yaml") as f:
    cfg = yaml.safe_load(f)

# Override for GPU
cfg["device"] = device
if device == "cuda":
    cfg["batch_size"] = 256
    cfg["epochs"] = 150
    cfg["lr"] = 0.0003
else:
    cfg["batch_size"] = 64
    cfg["epochs"] = 100
    cfg["lr"] = 0.0002

# Load dataset
_, smiles, labels = load_dataset_from_config(cfg)
print(f"Dataset: {len(smiles)} molecules")
print(f"Target range: [{min(labels):.4f}, {max(labels):.4f}]")
print(f"Config: epochs={cfg['epochs']}, batch_size={cfg['batch_size']}, lr={cfg['lr']}")

In [ ]:
# ── Cell 4: Build DataLoaders ─────────────────────────────────────────────
split_path = f"data/splits/{cfg['dataset']}/{cfg['split_type']}.npy"
train_idx, val_idx, test_idx = np.load(split_path, allow_pickle=True)

# Morgan fingerprints
morgan_cfg = cfg["morgan"]
morgan_feat = MorganFeaturizer(
    radius=morgan_cfg["radius"],
    n_bits=morgan_cfg["n_bits"],
    use_counts=morgan_cfg["use_counts"],
    use_chirality=morgan_cfg["use_chirality"],
)
fp_path = morgan_cfg["fp_path"]
if os.path.exists(fp_path):
    print(f"Loading pre-computed Morgan FPs from {fp_path}")
    morgan_fp_dict = MorganFeaturizer.load(fp_path)
else:
    print("Computing Morgan FPs on the fly...")
    morgan_fp_dict = None

# Graph featurizer
graph_feat = MolecularGraphFeaturizer()

# Dataset
full_dataset = MoleculeDataset(
    smiles_list=smiles,
    labels=labels,
    morgan_featurizer=morgan_feat,
    graph_featurizer=graph_feat,
    morgan_fp_dict=morgan_fp_dict,
)

batch_size = cfg["batch_size"]
use_pin = device.startswith("cuda")
num_workers = 4 if use_pin else 0

loaders = {}
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    subset = Subset(full_dataset, list(idx))
    loaders[name] = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=(name == "train"),
        collate_fn=collate_fn,
        pin_memory=use_pin,
        num_workers=num_workers,
        persistent_workers=(num_workers > 0),
    )

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

In [ ]:
# ── Cell 5: Helper functions ──────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None, use_amp=False):
    model.train()
    total_loss, n = 0.0, 0
    amp_device = "cuda" if device.startswith("cuda") else device
    for graphs, morgan_fps, labels_batch in loader:
        morgan_fps = morgan_fps.to(device, non_blocking=True)
        labels_batch = labels_batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=amp_device, enabled=use_amp):
            preds = model(graphs, morgan_fps)
            loss = criterion(preds, labels_batch)
        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        total_loss += loss.item() * morgan_fps.size(0)
        n += morgan_fps.size(0)
    return total_loss / n


def evaluate(model, loader, criterion, device, use_amp=False):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []
    amp_device = "cuda" if device.startswith("cuda") else device
    with torch.no_grad():
        for graphs, morgan_fps, labels_batch in loader:
            morgan_fps = morgan_fps.to(device, non_blocking=True)
            labels_batch = labels_batch.to(device, non_blocking=True)
            with autocast(device_type=amp_device, enabled=use_amp):
                preds = model(graphs, morgan_fps)
                loss = criterion(preds, labels_batch)
            total_loss += loss.item() * morgan_fps.size(0)
            n += morgan_fps.size(0)
            all_preds.append(preds.float().cpu().numpy())
            all_labels.append(labels_batch.float().cpu().numpy())
    all_preds = np.concatenate(all_preds).squeeze()
    all_labels = np.concatenate(all_labels).squeeze()
    return total_loss / n, all_preds, all_labels


def compute_metrics(preds, labels):
    """R2, RMSE, MAE, Pearson r."""
    r2 = r2_score(labels, preds)
    rmse = np.sqrt(mean_squared_error(labels, preds))
    mae = mean_absolute_error(labels, preds)
    pr, _ = pearsonr(preds, labels)
    return {"R2": r2, "RMSE": rmse, "MAE": mae, "Pearson_r": pr}


def top_k_recovery(preds, labels, k_pct):
    """Fraction of true top-k% compounds recovered in predicted top-k%."""
    n = len(labels)
    k = max(1, int(np.ceil(n * k_pct / 100.0)))
    true_top = set(np.argsort(labels)[-k:])
    pred_top = set(np.argsort(preds)[-k:])
    return len(true_top & pred_top) / len(true_top) * 100.0


def full_train(model, loaders, cfg, device, label="Model"):
    """Train model and return best model + all metrics."""
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
    criterion = nn.MSELoss()
    use_amp = device.startswith("cuda") and cfg.get("use_amp", True)
    scaler = GradScaler("cuda") if use_amp else None

    best_val_loss = float("inf")
    best_model = None
    epochs = cfg["epochs"]

    t0 = time.time()
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, loaders["train"], criterion, optimizer, device, scaler, use_amp)
        val_loss, _, _ = evaluate(model, loaders["val"], criterion, device, use_amp)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = deepcopy(model)
        if (epoch + 1) % 25 == 0 or epoch == 0:
            elapsed = time.time() - t0
            print(f"  [{label}] Epoch {epoch+1:>3}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Time: {elapsed:.0f}s")

    elapsed = time.time() - t0
    print(f"  [{label}] Training complete in {elapsed:.1f}s | Best Val Loss: {best_val_loss:.4f}")

    # Evaluate on all splits
    results = {}
    for split_name in ["train", "val", "test"]:
        _, preds, lbls = evaluate(best_model, loaders[split_name], criterion, device, use_amp)
        m = compute_metrics(preds, lbls)
        m["Top-10%"] = top_k_recovery(preds, lbls, 10)
        m["Top-20%"] = top_k_recovery(preds, lbls, 20)
        results[split_name] = m

    return best_model, results


print("Helper functions defined.")

## Model 1: Chemprop + Morgan Hybrid (ChemFF)</n\nThis is the full hybrid model: D-MPNN (300-dim) + Morgan FP (2048-dim) → FFN [1024→512→256→1]

In [ ]:
# ── Cell 6: Train Chemprop + Morgan Hybrid ────────────────────────────────
chemprop_cfg = cfg["chemprop"]
morgan_cfg   = cfg["morgan"]
ffn_cfg      = cfg["ffn"]

hybrid_model = ConcatFFN(
    chemprop_hidden_size=chemprop_cfg["hidden_size"],
    chemprop_depth=chemprop_cfg["depth"],
    chemprop_dropout=chemprop_cfg["dropout"],
    morgan_n_bits=morgan_cfg["n_bits"],
    ffn_hidden_layers=ffn_cfg["hidden_layers"],
    ffn_dropout=ffn_cfg["dropout"],
    output_dim=ffn_cfg["output_dim"],
).to(device)

n_params = sum(p.numel() for p in hybrid_model.parameters())
print(f"Chemprop + Morgan Hybrid — {n_params:,} parameters")
print("Training...")

hybrid_model, hybrid_results = full_train(
    hybrid_model, loaders, cfg, device, label="Chemprop+Morgan"
)

print("\n=== Chemprop + Morgan — Test Metrics ===")
for k, v in hybrid_results["test"].items():
    print(f"  {k}: {v:.4f}")

## Model 2: Chemprop-Only Baseline\n\nSame D-MPNN encoder but with Morgan FPs zeroed out, so the FFN head\noperates only on the 300-dim graph embedding (+ 2048 zeros).\nThis isolates the contribution of the graph encoder alone.

In [ ]:
# ── Cell 7: Train Chemprop-Only Baseline ──────────────────────────────────
# We create a wrapper that zeros out Morgan FPs to isolate D-MPNN contribution

class ChempropOnlyWrapper(nn.Module):
    """Wraps ConcatFFN but zeros out Morgan FPs → only graph encoder contributes."""
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model

    def forward(self, graphs, morgan_fps):
        # Zero out Morgan FPs so only Chemprop graph embedding is used
        zero_fps = torch.zeros_like(morgan_fps)
        return self.base(graphs, zero_fps)

    def parameters(self):
        return self.base.parameters()

# Fresh model for Chemprop-only
chemprop_only_inner = ConcatFFN(
    chemprop_hidden_size=chemprop_cfg["hidden_size"],
    chemprop_depth=chemprop_cfg["depth"],
    chemprop_dropout=chemprop_cfg["dropout"],
    morgan_n_bits=morgan_cfg["n_bits"],
    ffn_hidden_layers=ffn_cfg["hidden_layers"],
    ffn_dropout=ffn_cfg["dropout"],
    output_dim=ffn_cfg["output_dim"],
).to(device)

chemprop_only_model = ChempropOnlyWrapper(chemprop_only_inner)

n_params = sum(p.numel() for p in chemprop_only_model.parameters())
print(f"Chemprop-Only Baseline — {n_params:,} parameters")
print("Training...")

chemprop_only_model, chemprop_only_results = full_train(
    chemprop_only_model, loaders, cfg, device, label="Chemprop-Only"
)

print("\n=== Chemprop-Only — Test Metrics ===")
for k, v in chemprop_only_results["test"].items():
    print(f"  {k}: {v:.4f}")

## Results Summary\n\nThe tables below are formatted for direct copy-paste into the LaTeX progress report.

In [ ]:
# ── Cell 8: Summary Tables ────────────────────────────────────────────────

print("=" * 80)
print("RESULTS FOR PROGRESS REPORT — Copy these values into the LaTeX tables")
print("=" * 80)

# ── Chemprop-Only Baseline ──
ct = chemprop_only_results["test"]
print("\n┌─────────────────────────────────────────────────┐")
print("│  TABLE: Chemprop-Only Baseline (Test Set)       │")
print("├─────────────────────┬───────────────────────────┤")
print(f"│  R²                 │  {ct['R2']:.4f}                    │")
print(f"│  RMSE               │  {ct['RMSE']:.4f}                    │")
print(f"│  Pearson r          │  {ct['Pearson_r']:.4f}                    │")
print(f"│  Top-10% Recovery   │  {ct['Top-10%']:.2f}%                  │")
print(f"│  Top-20% Recovery   │  {ct['Top-20%']:.2f}%                  │")
print("└─────────────────────┴───────────────────────────┘")

# ── Chemprop + Morgan Hybrid ──
ht = hybrid_results["test"]
print("\n┌─────────────────────────────────────────────────┐")
print("│  TABLE: Chemprop + Morgan Hybrid (Test Set)     │")
print("├─────────────────────┬───────────────────────────┤")
print(f"│  R²                 │  {ht['R2']:.4f}                    │")
print(f"│  RMSE               │  {ht['RMSE']:.4f}                    │")
print(f"│  Pearson r          │  {ht['Pearson_r']:.4f}                    │")
print(f"│  Top-10% Recovery   │  {ht['Top-10%']:.2f}%                  │")
print(f"│  Top-20% Recovery   │  {ht['Top-20%']:.2f}%                  │")
print("└─────────────────────┴───────────────────────────┘")

# ── Relative improvement ──
print("\n┌─────────────────────────────────────────────────────────────────┐")
print("│  TABLE: Comparison (Relative Δ)                                │")
print("├───────────────────┬──────────┬──────────┬──────────────────────┤")
print("│  Metric           │ Baseline │  Hybrid  │  Relative Δ         │")
print("├───────────────────┼──────────┼──────────┼──────────────────────┤")
r2_delta = (ht['R2'] - ct['R2']) / abs(ct['R2']) * 100 if ct['R2'] != 0 else float('inf')
rmse_delta = (ht['RMSE'] - ct['RMSE']) / ct['RMSE'] * 100
pr_delta = (ht['Pearson_r'] - ct['Pearson_r']) / ct['Pearson_r'] * 100
t10_delta = (ht['Top-10%'] - ct['Top-10%']) / ct['Top-10%'] * 100 if ct['Top-10%'] != 0 else float('inf')
t20_delta = (ht['Top-20%'] - ct['Top-20%']) / ct['Top-20%'] * 100 if ct['Top-20%'] != 0 else float('inf')
print(f"│  R²               │  {ct['R2']:.4f} │  {ht['R2']:.4f} │  {r2_delta:+.1f}%               │")
print(f"│  RMSE             │  {ct['RMSE']:.4f} │  {ht['RMSE']:.4f} │  {rmse_delta:+.1f}%               │")
print(f"│  Pearson r        │  {ct['Pearson_r']:.4f} │  {ht['Pearson_r']:.4f} │  {pr_delta:+.1f}%               │")
print(f"│  Top-10% Recovery │  {ct['Top-10%']:.2f}%│  {ht['Top-10%']:.2f}%│  {t10_delta:+.1f}%               │")
print(f"│  Top-20% Recovery │  {ct['Top-20%']:.2f}%│  {ht['Top-20%']:.2f}%│  {t20_delta:+.1f}%               │")
print("└───────────────────┴──────────┴──────────┴──────────────────────┘")

# ── Cross-pipeline comparison (including Sushmita's results) ──
print("\n┌────────────────────────────────────────────────────────────────────────────┐")
print("│  TABLE: Cross-Pipeline Comparison (for LaTeX Table 6)                     │")
print("├──────────────────────────┬────────┬────────┬──────────┬──────────────────┤")
print("│  Model                   │   R²   │  RMSE  │ Pearson r│  Top-10%         │")
print("├──────────────────────────┼────────┼────────┼──────────┼──────────────────┤")
print(f"│  KPGT-only (baseline)    │ 0.6691 │ 1.9159 │   0.8205 │  54.55%          │")
print(f"│  KPGT + Morgan (MoFPGNN)│ 0.7346 │ 1.7172 │   0.8611 │  81.82%          │")
print(f"│  Chemprop-only (baselin)│ {ct['R2']:.4f} │ {ct['RMSE']:.4f} │   {ct['Pearson_r']:.4f} │  {ct['Top-10%']:.2f}%          │")
print(f"│  Chemprop + Morgan      │ {ht['R2']:.4f} │ {ht['RMSE']:.4f} │   {ht['Pearson_r']:.4f} │  {ht['Top-10%']:.2f}%          │")
print(f"│  LANTERN KPGT (reported)│ ~0.66  │   ---  │     ---  │     ---          │")
print("└──────────────────────────┴────────┴────────┴──────────┴──────────────────┘")

# ── All-split details ──
print("\n\n=== FULL SPLIT BREAKDOWN (for reference) ===")
for model_name, results in [("Chemprop-Only", chemprop_only_results), ("Chemprop+Morgan", hybrid_results)]:
    print(f"\n--- {model_name} ---")
    for split in ["train", "val", "test"]:
        r = results[split]
        print(f"  {split:>5}: R²={r['R2']:.4f}  RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}  "
              f"Pearson={r['Pearson_r']:.4f}  Top10={r['Top-10%']:.1f}%  Top20={r['Top-20%']:.1f}%")

In [ ]:
# ── Cell 9: LaTeX-ready snippet ───────────────────────────────────────────
# Copy-paste this directly into the .tex file

ct = chemprop_only_results["test"]
ht = hybrid_results["test"]

r2_delta = (ht['R2'] - ct['R2']) / abs(ct['R2']) * 100 if ct['R2'] != 0 else 0
rmse_delta = (ht['RMSE'] - ct['RMSE']) / ct['RMSE'] * 100
pr_delta = (ht['Pearson_r'] - ct['Pearson_r']) / ct['Pearson_r'] * 100
t10_delta = (ht['Top-10%'] - ct['Top-10%']) / ct['Top-10%'] * 100 if ct['Top-10%'] != 0 else 0
t20_delta = (ht['Top-20%'] - ct['Top-20%']) / ct['Top-20%'] * 100 if ct['Top-20%'] != 0 else 0

# Determine which values are better for bolding
def best(a, b, higher_better=True):
    if higher_better:
        return (r"\\textbf{" + f"{a:.4f}" + "}", f"{b:.4f}") if a > b else (f"{a:.4f}", r"\\textbf{" + f"{b:.4f}" + "}")
    else:
        return (r"\\textbf{" + f"{a:.4f}" + "}", f"{b:.4f}") if a < b else (f"{a:.4f}", r"\\textbf{" + f"{b:.4f}" + "}")

print("=" * 80)
print("LaTeX for TABLE: Chemprop-Only Baseline (replace Table X placeholder)")
print("=" * 80)
print(f"""
$R^2$                & {ct['R2']:.4f} \\\\
RMSE                 & {ct['RMSE']:.4f} \\\\
Pearson $r$          & {ct['Pearson_r']:.4f} \\\\
Top-10\\% Recovery    & {ct['Top-10%']:.2f}\\% \\\\
Top-20\\% Recovery    & {ct['Top-20%']:.2f}\\% \\\\
""")

print("=" * 80)
print("LaTeX for TABLE: Chemprop + Morgan Hybrid")
print("=" * 80)
print(f"""
$R^2$                & {ht['R2']:.4f} \\\\
RMSE                 & {ht['RMSE']:.4f} \\\\
Pearson $r$          & {ht['Pearson_r']:.4f} \\\\
Top-10\\% Recovery    & {ht['Top-10%']:.2f}\\% \\\\
Top-20\\% Recovery    & {ht['Top-20%']:.2f}\\% \\\\
""")

print("=" * 80)
print("LaTeX for TABLE: Comparison")
print("=" * 80)
print(f"""
$R^2$ $\\uparrow$            & {ct['R2']:.4f} & {ht['R2']:.4f} & ${r2_delta:+.1f}\\%$  \\\\
RMSE $\\downarrow$           & {ct['RMSE']:.4f} & {ht['RMSE']:.4f} & ${rmse_delta:+.1f}\\%$ \\\\
Pearson $r$ $\\uparrow$      & {ct['Pearson_r']:.4f} & {ht['Pearson_r']:.4f} & ${pr_delta:+.1f}\\%$  \\\\
Top-10\\% Recovery $\\uparrow$& {ct['Top-10%']:.2f}\\% & {ht['Top-10%']:.2f}\\% & ${t10_delta:+.1f}\\%$ \\\\
Top-20\\% Recovery $\\uparrow$& {ct['Top-20%']:.2f}\\% & {ht['Top-20%']:.2f}\\% & ${t20_delta:+.1f}\\%$ \\\\
""")

print("=" * 80)
print("LaTeX for TABLE: Cross-Pipeline (Table 6)")
print("=" * 80)
print(f"""
KPGT-only (baseline)     & 0.6691 & 1.9159 & 0.8205 & 54.55\\% \\\\
KPGT + Morgan (MoFPGNN)  & 0.7346 & 1.7172 & 0.8611 & 81.82\\% \\\\
\\midrule
Chemprop-only (baseline) & {ct['R2']:.4f} & {ct['RMSE']:.4f} & {ct['Pearson_r']:.4f} & {ct['Top-10%']:.2f}\\% \\\\
Chemprop + Morgan        & {ht['R2']:.4f} & {ht['RMSE']:.4f} & {ht['Pearson_r']:.4f} & {ht['Top-10%']:.2f}\\% \\\\
\\midrule
LANTERN KPGT (reported)  & $\\approx 0.66$ & --- & --- & --- \\\\
""")